In [1]:
import re

from ndpi.visualization.tables import (
    escape_latex,
    fix_gt_table,
    raw_latex,
    stacked_header,
)

from ech_discovery.data.aggregates import common
from ech_discovery.data.enrichment.ech import (
    EXAMPLE_PUBLIC_NAME_MERGED,
    config_field,
    is_example_public_name,
    is_nordvpn_target,
    replace_nordvpn_public_name,
)
from ech_discovery.notebook import *

In [2]:
cfg_public_names = config_field(C.ech_public_name, C.retry_configs_considered_unique)
cfg_max_name_len = config_field(C.ech_max_name_len, C.retry_configs_considered_unique)

df_ech = (
    common.ech_with_dns_status(
        limit_per_location=True, regex_filter="tud/2026-07-30", limit=1
    )
    .with_columns(
        # Reduce ECH configuration to ciphersuite, ech_public name, and the order
        pl.col(C.retry_configs)
        .list.eval(
            pl.struct(
                pl.element().struct.field(
                    C.ech_public_name, C.ech_cipher_suites, C.ech_max_name_len
                )
            )
        )
        .alias(C.retry_configs_considered_unique)
    )
    .with_columns(
        pl.when(
            is_nordvpn_target(),
            is_example_public_name(),
            # Ensure all ech-public names replaced match exactly the used ech_public_name
            cfg_public_names.list.min() == pl.col(C.ech_public_name),
            cfg_public_names.list.max() == pl.col(C.ech_public_name),
        )
        .then(
            pl.col(C.retry_configs_considered_unique).list.eval(
                pl.element().struct.with_fields(
                    pl.lit(EXAMPLE_PUBLIC_NAME_MERGED).alias(C.ech_public_name)
                )
            )
        )
        .otherwise(pl.col(C.retry_configs_considered_unique)),
        # Max Name Length
        cfg_max_name_len.list.min().alias(C.ech_max_name_len_min),
        cfg_max_name_len.list.max().alias(C.ech_max_name_len_max),
    )
    .pipe(replace_nordvpn_public_name, C.target)
)

df_variations = pl.read_parquet(
    config.PROCESSED_DATA_DIR / "echtool_configuration_variations.pq.zst"
)
df_rotations = pl.read_parquet(
    config.PROCESSED_DATA_DIR
    / "echtool_config_rotation_by_public_name_graceperiod_cert_noverify.pq.zst"
)

assert (
    df_rotations[C.ech_public_name].null_count() == 0
), "null-keyed rotation rows never join"

In [3]:
# Per deployment: the value shown in the table, and the column asserting it is unique
config_shape = {
    Strings.configurations: (
        pl.col(C.retry_configs_considered_unique).list.len(),
        C.retry_configs_n_unique,
    ),
    Strings.ech_public_names: (
        cfg_public_names.list.n_unique(),
        C.retry_configs_public_names_n_unique,
    ),
    Strings.ech_public_keys: (
        config_field(C.ech_public_key).list.n_unique(),
        C.retry_configs_public_keys_n_unique,
    ),
}

pagg = (
    df_ech.group_by(C.ech_public_name, C.retry_configs_considered_unique)
    .agg(
        pl.col(C.target_sld).n_unique(),
        pl.when(pl.col(Strings.ech_in_dns).all())
        .then(pl.lit(Strings.coverage_all))
        .when(pl.col(Strings.ech_in_dns).any())
        .then(pl.lit(Strings.coverage_partial))
        .otherwise(pl.lit(Strings.coverage_none))
        .alias(Strings.ech_in_dns),
        pl.when((pl.col(C.role) == "ech").all())
        .then(pl.lit([Strings.transport_tcp.value]))
        .when((pl.col(C.role) == "ech-quic").all())
        .then(pl.lit([Strings.transport_quic.value]))
        .when((pl.col(C.role) == "ech").any() & (pl.col(C.role) == "ech-quic").any())
        .then(pl.lit([Strings.transport_tcp.value, Strings.transport_quic.value]))
        .otherwise(pl.lit(["error"]))
        .alias(Strings.transport),
        *(expr.max().alias(name) for name, (expr, _) in config_shape.items()),
        pl.when(
            pl.col(C.ech_max_name_len_min).first()
            == pl.col(C.ech_max_name_len_max).first()
        )
        .then(pl.col(C.ech_max_name_len_max).first().cast(pl.String))
        .otherwise(
            pl.col(C.ech_max_name_len_min).first().cast(pl.String)
            + pl.lit("-")
            + pl.col(C.ech_max_name_len_max).first().cast(pl.String)
        )
        .alias(Strings.ech_max_name_len),
        *(expr.n_unique().alias(check) for _, (expr, check) in config_shape.items()),
    )
    .sort([C.target_sld, C.ech_public_name], descending=[True, False])
    .collect()
)
pagg = (
    pagg.join(df_variations, on=C.ech_public_name, validate="m:1", how="left")
    .drop(C.measurement)
    .join(df_rotations, on=C.ech_public_name, how="left", validate="m:1")
).with_columns(
    cs.contains("[h]").fill_null(Strings.latex_xmark),
)

In [4]:
for name, (_, check) in config_shape.items():
    assert pagg[check].eq(1).all(), f"{name} are not identical within a deployment"

In [5]:
common.ech_with_dns_status(
    limit_per_location=True, regex_filter="tud/2026-07-30", limit=1
).filter(
    pl.col(C.ech_public_name) != "cloudflare-ech.com",
    pl.col(C.ech_public_name) != "graph.facebook.com",
).group_by(
    Strings.ech_in_dns
).agg(
    pl.col(C.target_sld).n_unique()
).collect()

ECH in DNS,target-sld
bool,u32
false,48
true,143


In [6]:
common.ech_with_dns_status(
    limit_per_location=True, regex_filter="tud/2026-07-30", limit=1
).filter(
    pl.col(C.ech_public_name) != "cloudflare-ech.com",
    pl.col(C.ech_public_name) != "graph.facebook.com",
).group_by(
    Strings.ech_in_dns
).agg(
    pl.col(C.target).n_unique()
).collect()

ECH in DNS,target
bool,u32
false,72
true,"2,597"


# Table output

In [7]:
mapping = {
    EchtoolErrorKind.accepted: Strings.latex_cmark,
    EchtoolErrorKind.ech_rejected_with_retry: Strings.latex_retry_config,
    EchtoolErrorKind.tls_alert: Strings.latex_tls_alert,
    EchtoolErrorKind.quic_invalid_token: Strings.quic_error,
    EchtoolErrorKind.no_retry_config: Strings.latex_xmark,
}

expr = pl.element()
for old, new in mapping.items():
    expr = pl.when(pl.element() == old).then(pl.lit(new)).otherwise(expr)


table_data = (
    pagg.rename(
        {
            C.ech_public_name: Strings.ech_public_name,
            C.target_sld: Strings.second_level_domains,
            Strings.ech_in_dns: Strings.ech_in_https,
            C.mean_grace_period: Strings.ech_key_grace_period,
            C.rotation_mean_retry: Strings.ech_key_rotation_interval,
            C.ech_config_config_id: Strings.ech_config_id,
            C.outer_ech_config_id: Strings.outer_ech_config_id,
            C.outer_sni: Strings.outer_sni,
        }
    )
    .with_columns(cs.list().list.sort().list.eval(expr))
    .with_columns(
        pl.col(
            Strings.ech_key_grace_period, Strings.ech_key_rotation_interval
        ).fill_null(Strings.latex_xmark)
    )
    .drop(
        cs.contains("_"),
        Strings.ech_config_id,
    )
)


OUTER_COLS = [Strings.outer_ech_config_id, Strings.outer_sni, Strings.transport]
new_cols = {
    col: table_data.explode(col)[col].drop_nulls().unique().sort().to_list()
    for col in OUTER_COLS
}

table_data = table_data.with_columns(
    [
        pl.when(pl.col(col).list.contains(val))
        .then(pl.lit(Strings.latex_cmark))
        .when(pl.col(col).list.len() > 0)
        .then(pl.lit(Strings.latex_xmark))
        .otherwise(pl.lit(Strings.not_available))
        .alias(f"{col} {val}")
        for col, vals in new_cols.items()
        for val in vals
    ]
).drop(OUTER_COLS)
span_cols = {col: [f"{col} {val}" for val in vals] for col, vals in new_cols.items()}
shift = [Strings.ech_key_rotation_interval, Strings.ech_key_grace_period]
table_data = table_data.select(pl.exclude(shift), *shift)

/tmp/ipykernel_1123713/1774820900.py:42: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  col: table_data.explode(col)[col].drop_nulls().unique().sort().to_list()


In [8]:
table_data = table_data.filter(pl.col(Strings.second_level_domains) > 1)
nunique = table_data.select(
    cs.string().replace(Strings.not_available, None).drop_nulls().n_unique(),
    (~cs.string()).n_unique(),
)
cols_to_keep = [c for c in table_data.columns if nunique[c][0] > 1]
table_data = table_data.select(cols_to_keep).rename(
    {Strings.configurations: Strings.configurations_wrapped}
)

In [9]:
ECH_SYMBOL = {
    Strings.coverage_all: Strings.latex_cmark.value,
    Strings.coverage_partial: Strings.latex_cmark_partial.value,
    Strings.coverage_none: Strings.latex_xmark.value,
}


def present(columns):
    return [col for col in columns if col in table_data.columns]


variation_labels = {
    f"{col} {val}": val for col, vals in new_cols.items() for val in vals
}

WRAPPED_HEADERS = [
    Strings.second_level_domains,
    Strings.ech_in_https,
    Strings.configurations_wrapped,
    Strings.ech_public_names,
    Strings.ech_public_keys,
    Strings.ech_max_name_len,
    Strings.ech_key_rotation_interval,
    Strings.ech_key_grace_period,
    Strings.outer_ech_config_id,
    Strings.outer_sni,
    Strings.transport,
]

caption = raw_latex(
    r"ECH configuration of deployments visible on July 30, 2026 with at least 2 second-level domains. "
    r"Most deployments do not rotate key material at regular intervals. "
    r"We show whether ECH configuration can be found \cmark, partially found (\cmark), or not found \xmark~in HTTPS RRs. "
    r"2nd-level domains are counted with respect to the Mozilla Public Suffix List~\cite{m-psl-07}. "
    r"We test modifications of the outer ClientHello compared to the provided ECH configuration. "
    r"We indicate acceptance \cmark, receiving a Retry configuration $\circlearrowleft$ and TLS alerts \textcircled{\small !} from at least one probed domain. "
    r"Assessing Key rotation and grace periods we denote an interval if we are able to infer it from the obtained RetryConfigurations. "
    r"Whenever we observe rotation, but do not have enough samples we denote \cmark instead of providing an interval. "
    r"We denote no observed rotation or grace period with \xmark. "
    r"\textit{n/a} denotes when none of the domains allowed a connection at the time of measurement. "
    r"Note, a single ECH public name can be used by multiple distinct deployments (e.g., google.com). "
    r"The table presents the maximum number in ECH Configurations columns. "
    r"We present at a key rotation interval if at least one of the domains does regular rotation. "
)

gt = (
    GT(table_data)
    .tab_header(title=caption)
    .fmt(fns=escape_latex, columns=[Strings.ech_public_name])
    .fmt_number(
        columns=Strings.second_level_domains,
        compact=True,
        decimals=1,
        drop_trailing_zeros=True,
    )
    .tab_spanner(
        Strings.outer_ech_config_id,
        columns=present(span_cols[Strings.outer_ech_config_id]),
        id="span_ech_id",
    )
    .tab_spanner(
        Strings.outer_sni,
        columns=present(span_cols[Strings.outer_sni]),
        id="span_sni",
    )
    .tab_spanner(Strings.outer_ch_variation, spanners=["span_ech_id", "span_sni"])
    .tab_spanner(
        Strings.ech_configurations,
        columns=[
            Strings.configurations_wrapped,
            Strings.ech_public_names,
            Strings.ech_public_keys,
            Strings.ech_max_name_len,
        ],
    )
    .tab_spanner(
        Strings.general,
        columns=[
            Strings.second_level_domains,
            Strings.ech_in_https,
            Strings.transport_tcp_col,
            Strings.transport_quic_col,
        ],
    )
    .tab_spanner(
        Strings.ech_key,
        columns=[Strings.ech_key_rotation_interval, Strings.ech_key_grace_period],
    )
    .fmt(fns=lambda x: ECH_SYMBOL[x], columns=Strings.ech_in_https)
    .fmt(
        fns=lambda x: x,
        columns=present(
            [c for cols in span_cols.values() for c in cols]
            + [Strings.ech_key_rotation_interval, Strings.ech_key_grace_period]
        ),
    )
    .cols_align(align="right", columns=table_data.columns[1:])
)


tbl = (
    re.sub(
        r"\\\\([a-z])",
        r"\\\1",
        fix_gt_table(
            gt.cols_label(
                **{
                    header: stacked_header(header, "1.2cm", valign="t")
                    for header in present(WRAPPED_HEADERS)
                },
                **{
                    col: label
                    for col, label in variation_labels.items()
                    if col in table_data.columns
                },
            ).as_latex(),
            label="tab:ech-deployment-configuration",
        ),
    )
    .replace("\\$", "$")
    .replace("\\{", "{")
    .replace("\\}", "}")
    .replace(
        " & \\multicolumn{4}{c}{Outer CH Variation} &  \\\\ \n\\cmidrule(lr){2-5}",
        " & \\multicolumn{8}{c}{} & \\multicolumn{4}{c}{Outer CH Variation} & \\multicolumn{2}{c}{} \\\\ \n\\cmidrule(lr){10-13}",
    )
    .replace(
        "\\begin{tabular}{lrrrrrrrrrrrrrr}",  # 14 r's, was 15
        "\\begin{tabular}{lrrrrrrrr"
        + (">{\\centering\\arraybackslash}p{0.8cm}" * 2)
        + "rrrr}",
    )
)
display(gt)
print(tbl)

GT(_tbl_data=shape: (30, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ ECH       ┆ Second-Le ┆ ECH in    ┆ Config-   ┆ … ┆ Transport ┆ Transport ┆ Rotation  ┆ Grace    │
│ Public    ┆ vel       ┆ HTTPS RRs ┆ urations  ┆   ┆ QUIC      ┆ TCP       ┆ Interval  ┆ Period   │
│ Name      ┆ Domains   ┆ ---       ┆ [#]       ┆   ┆ ---       ┆ ---       ┆ [h]       ┆ [h]      │
│ ---       ┆ [#]       ┆ str       ┆ ---       ┆   ┆ str       ┆ str       ┆ ---       ┆ ---      │
│ str       ┆ ---       ┆           ┆ u32       ┆   ┆           ┆           ┆ str       ┆ str      │
│           ┆ u32       ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ cloudflar ┆ 4,233,198 ┆ partial   ┆ 1         ┆ … ┆ \cmark    ┆ \cmark    ┆ 1         ┆ 4        │
│ e-ech.com ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ graph.fac ┆ 4,677     ┆ none      ┆ 10        ┆ … ┆ \cmark    ┆ \cmark    ┆ \xmark    ┆ \xmark   │
│ ebook.com ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ ech.encry ┆ 44        ┆ partial   ┆ 1         ┆ … ┆ \cmark    ┆ \cmark    ┆ \xmark    ┆ \xmark   │
│ ptedclien ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ thello.co ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ m         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ …         ┆ …         ┆ …         ┆ …         ┆ … ┆ …         ┆ …         ┆ …         ┆ …        │
│ sky.xn--1 ┆ 2         ┆ all       ┆ 1         ┆ … ┆ \xmark    ┆ \cmark    ┆ \xmark    ┆ \xmark   │
│ xa.pl     ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ wmcurrenc ┆ 2         ┆ partial   ┆ 1         ┆ … ┆ \xmark    ┆ \cmark    ┆ \xmark    ┆ \xmark   │
│ y.exchang ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ e         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x74503f2530e0>, _boxhead=Boxhead([ColInfo(var='ECH Public Name', type=<ColInfoTypeEnum.default: 1>, column_label='ECH Public Name', column_align='left', column_width=None), ColInfo(var='Second-Level Domains [#]', type=<ColInfoTypeEnum.default: 1>, column_label='Second-Level Domains [#]', column_align='right', column_width=None), ColInfo(var='ECH in HTTPS RRs', type=<ColInfoTypeEnum.default: 1>, column_label='ECH in HTTPS RRs', column_align='right', column_width=None), ColInfo(var='Transport TCP', type=<ColInfoTypeEnum.default: 1>, column_label='Transport TCP', column_align='right', column_width=None), ColInfo(var='Transport QUIC', type=<ColInfoTypeEnum.default: 1>, column_label='Transport QUIC', column_align='right', column_width=None), ColInfo(var='Config- urations [#]', type=<ColInfoTypeEnum.default: 1>, column_label='Config- urations [#]', column_align='right', column_width=None), ColInfo(var='Public Names [#]', type=<ColInfoTypeEnum.default: 1>, column_label='Public Names [#]', column_align='right', column_width=None), ColInfo(var='Public Keys [#]', type=<ColInfoTypeEnum.default: 1>, column_label='Public Keys [#]', column_align='right', column_width=None), ColInfo(var='Max Name Length [chars]', type=<ColInfoTypeEnum.default: 1>, column_label='Max Name Length [chars]', column_align='right', column_width=None), ColInfo(var='ECH Config ID $\\circlearrowleft$', type=<ColInfoTypeEnum.default: 1>, column_label='ECH Config ID $\\circlearrowleft$', column_align='right', column_width=None), ColInfo(var='ECH Config ID \\textcircled{\\small !}', type=<ColInfoTypeEnum.default: 1>, co

\begin{table*}[!t]
\caption{ECH configuration of deployments visible on July 30, 2026 with at least 2 second-level domains. Most deployments do not rotate key material at regular intervals. We show whether ECH configuration can be found \cmark, partially found (\cmark), or not found \xmark~in HTTPS RRs. 2nd-level domains are counted with respect to the Mozilla Public Suffix List~\cite{m-psl-07}. We test modifications of the outer ClientHello compared to the provided ECH configuration. We indicate acceptance \cmark, receiving a Retry configuration $\circlearrowleft$ and TLS alerts \textcircled{\small !} from at least one probed domain. Assessing Key rotation and grace periods we denote an interval if we are able to infer it from the obtained RetryConfigurations. Whenever we observe rotation, but do not have enough samples we denote \cmark instead of providing an interval. We denote no observed rotation or grace period with \xmark. \textit{n/a} denotes when none of the domains allowed a 